# Week 78, Day 4: Web scraping, responsibly

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w078d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from urllib.parse import urljoin, urldefrag

page = "https://books.example/catalogue/page-2.html"
for href in ["page-3.html", "../index.html", "/about", "#reviews", "https://other.example/x", "item.html#top"]:
    url = urljoin(page, href)
    print(f"{href:<26}{urldefrag(url)[0]}")

In [ ]:
from urllib.robotparser import RobotFileParser

robots_txt = """
User-agent: *
Allow: /private/press
Disallow: /private
Disallow: /search
Crawl-delay: 2

User-agent: AtlasBot
Disallow: /drafts
Crawl-delay: 5
"""
rp = RobotFileParser()
rp.parse(robots_txt.splitlines())
for agent in ("AtlasBot/1.0", "SomeOtherBot"):
    allowed = [path for path in ("/", "/private/a", "/private/press/1", "/drafts/1", "/search?q=x")
               if rp.can_fetch(agent, "https://site.example" + path)]
    print(f"{agent:<14}delay {rp.crawl_delay(agent)}  allowed {allowed}")

In [ ]:
from html.parser import HTMLParser

class Headlines(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_h2 = False
        self.headlines = []
        self.text = ""

    def handle_starttag(self, tag, attrs):
        if tag == "h2" and ("class", "headline") in attrs:
            self.in_h2, self.text = True, ""

    def handle_endtag(self, tag):
        if tag == "h2" and self.in_h2:
            self.headlines.append(" ".join(self.text.split()))
            self.in_h2 = False

    def handle_data(self, data):
        if self.in_h2:
            self.text += data

page = """<main>
  <h2 class="headline">Trains <em>back</em> on time</h2>
  <h2>Not a headline</h2>
  <h2 class="headline">
     Fish &amp; chips prices rise
  </h2></main>"""
p = Headlines()
p.feed(page)
p.close()
print(p.headlines)

> **Key idea.** Scrape only when there's no API, download or feed; check the terms, robots.txt and copyright; avoid personal data (GDPR); keep the load low; identify your bot with a User-Agent and contact address. urllib.robotparser reads robots.txt: parse(lines), can_fetch(agent, url), crawl_delay(agent); a group naming your bot replaces the * group, and Python uses the first matching rule. html.parser.HTMLParser calls handle_starttag(tag, attrs), handle_endtag and handle_data; track state, add text pieces together and normalise whitespace. Resolve links with urljoin, strip fragments with urldefrag, skip mailto/tel/javascript, and compare hosts with urlsplit. Prefer JSON-LD and other structured data to visible text. A polite crawler keeps a queue and a seen set, stays on one site within a page limit, obeys robots.txt and delays, backs off on 429 and 503, parses only HTML, and caches. Test parsers against saved pages.

### Exercise: Every link on a page

Write `links(html, page_url)`: every link on the page as a sorted list of unique absolute URLs. Take the `href` of each `<a>` tag (ignoring `<a>` tags without one), resolve it against `page_url` with `urljoin`, remove any fragment, and keep only `http` and `https` URLs (so `mailto:`, `tel:` and `javascript:` links disappear). Then `same_site(urls, page_url)`: those whose host (`netloc`, compared in lower case) is the same as the page's.

In [ ]:
from html.parser import HTMLParser
from urllib.parse import urljoin, urldefrag, urlsplit

def links(html, page_url):
    ___

def same_site(urls, page_url):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PAGE = \"\"\"<nav><a href=\"/\">Home</a> <a href=\"about.html#team\">About</a> <a href=\"mailto:hi@shop.example\">Mail</a></nav>\n<p>See <a href=\"../catalogue/item-7.html\">item 7</a>, <a href=\"https://SHOP.example/sale\">the sale</a>,\n<a href=\"https://blog.example/post\">our blog</a>, <a name=\"anchor\">no link</a>, <a href=\"javascript:void(0)\">menu</a>\nand <a href='about.html'>About again</a>.</p><a href=\"tel:+441234\">Call</a>\"\"\"\n\ndef test_links():\n    \"resolved, deduplicated, web links only\"\n    got = links(PAGE, \"https://shop.example/pages/info.html\")\n    assert got == [\"https://SHOP.example/sale\", \"https://blog.example/post\", \"https://shop.example/\",\n                   \"https://shop.example/catalogue/item-7.html\", \"https://shop.example/pages/about.html\"]\n\ndef test_same_site():\n    \"the same host\"\n    got = same_site(links(PAGE, \"https://shop.example/pages/info.html\"), \"https://shop.example/pages/info.html\")\n    assert got == [\"https://SHOP.example/sale\", \"https://shop.example/\", \"https://shop.example/catalogue/item-7.html\",\n                   \"https://shop.example/pages/about.html\"]")

### Exercise: A table into rows

Write `parse_table(html, table_id)`: find the `<table>` with that `id` and return its rows as a list of dicts, using the text of the `<th>` cells in its first row as keys. Cell text is all the text inside the cell (including inside nested tags such as `<a>` or `<b>`), with whitespace collapsed to single spaces and trimmed. Ignore other tables on the page, rows with no `<td>` cells, and a `<td>` that's shorter than the header (missing cells become `""`). Raise `LookupError(f"no table {table_id}")` if it isn't there.

In [ ]:
from html.parser import HTMLParser

def parse_table(html, table_id):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PAGE = \"\"\"<table id=\"other\"><tr><th>X</th></tr><tr><td>nope</td></tr></table>\n<table id=\"trains\" class=\"times\">\n  <tr><th>Train</th><th>Departs</th> <th>Platform</th></tr>\n  <tr><td><a href=\"/t/1\">London   Euston</a></td><td>09:12</td><td>  4 </td></tr>\n  <tr><td>Glasgow <b>Central</b></td><td>09:30</td></tr>\n  <tr><td colspan=\"3\"></td></tr>\n  <tr><td>Fish &amp; Chip   Express</td><td>10:00</td><td>12</td></tr>\n</table>\"\"\"\n\ndef test_table():\n    \"header keys, nested text, missing cells\"\n    assert parse_table(PAGE, \"trains\") == [\n        {\"Train\": \"London Euston\", \"Departs\": \"09:12\", \"Platform\": \"4\"},\n        {\"Train\": \"Glasgow Central\", \"Departs\": \"09:30\", \"Platform\": \"\"},\n        {\"Train\": \"\", \"Departs\": \"\", \"Platform\": \"\"},\n        {\"Train\": \"Fish & Chip Express\", \"Departs\": \"10:00\", \"Platform\": \"12\"}]\n    assert parse_table(PAGE, \"other\") == [{\"X\": \"nope\"}]\n\ndef test_missing():\n    \"a clear error\"\n    try:\n        parse_table(PAGE, \"buses\")\n    except LookupError as e:\n        assert str(e) == \"no table buses\"\n    else:\n        raise AssertionError(\"expected LookupError\")")

### Exercise: Ask robots.txt first

Write `allowed_urls(robots_txt, agent, urls)`: the URLs (in their original order) that `agent` may fetch according to the `robots_txt` text, using `urllib.robotparser`. Then `delay_for(robots_txt, agent, default=1.0)`: the crawl delay for that agent (as a `float`), or `default` if robots.txt doesn't give one. And `robots_url(page_url)`: the address of the robots.txt file for any page's site (same scheme and host, path `/robots.txt`).

In [ ]:
from urllib.parse import urlsplit, urlunsplit
from urllib.robotparser import RobotFileParser

def allowed_urls(robots_txt, agent, urls):
    ___

def delay_for(robots_txt, agent, default=1.0):
    ___

def robots_url(page_url):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("ROBOTS = \"\"\"# a comment\nUser-agent: *\nDisallow: /admin\nDisallow: /cart\nCrawl-delay: 3\n\nUser-agent: Scrappy\nDisallow: /\n\"\"\"\n\ndef test_allowed():\n    \"rules by agent\"\n    urls = [\"https://s.example/\", \"https://s.example/admin/users\", \"https://s.example/cart?x=1\", \"https://s.example/products/7\"]\n    assert allowed_urls(ROBOTS, \"AtlasBot/1.0\", urls) == [\"https://s.example/\", \"https://s.example/products/7\"]\n    assert allowed_urls(ROBOTS, \"Scrappy/2\", urls) == []\n    assert allowed_urls(\"\", \"AtlasBot\", urls) == urls\n\ndef test_delay_and_url():\n    \"delay and location\"\n    assert delay_for(ROBOTS, \"AtlasBot\") == 3.0 and isinstance(delay_for(ROBOTS, \"AtlasBot\"), float)\n    assert delay_for(ROBOTS, \"Scrappy\", 2.5) == 2.5 and delay_for(\"\", \"x\") == 1.0\n    assert robots_url(\"https://shop.example:8443/a/b.html?q=1#x\") == \"https://shop.example:8443/robots.txt\"")

### Exercise: Product cards

A shop lists products as cards like `<div class="product" data-sku="A12"><h3>Name</h3><span class="price">£12.50</span></div>` (the `class` may hold several classes, such as `"product featured"`). Write `parse_products(html)`: a list of dicts `{"sku": ..., "name": ..., "pence": ...}`, one per card in page order, where `name` is the `<h3>`'s text (whitespace collapsed) and `pence` is the price as a whole number of pence (`"£1,299.99"` gives `129999`), or `None` if the card has no price or it can't be read (`"Sold out"`). Ignore prices outside product cards.

In [ ]:
import re
from html.parser import HTMLParser

def parse_products(html):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PAGE = \"\"\"<span class=\"price\">\u00a31.00</span>\n<div class=\"product featured\" data-sku=\"K-1\"><h3>Kettle <small>(1.7 l)</small></h3>\n  <p>Fast.</p><span class=\"price old\">\u00a330.00</span></div>\n<div class=\"product\" data-sku=\"T-9\"><h3>  Toaster </h3><span class=\"price\">\u00a31,299.99</span></div>\n<div class=\"productive\" data-sku=\"X\"><h3>Not a card</h3></div>\n<div class=\"product\" data-sku=\"M-2\"><h3>Mug</h3><span class=\"price\">Sold out</span></div>\n<div class=\"product\" data-sku=\"S-5\"><h3>Spoon</h3></div>\"\"\"\n\ndef test_products():\n    \"cards, names and prices in pence\"\n    assert parse_products(PAGE) == [\n        {\"sku\": \"K-1\", \"name\": \"Kettle (1.7 l)\", \"pence\": 3000},\n        {\"sku\": \"T-9\", \"name\": \"Toaster\", \"pence\": 129999},\n        {\"sku\": \"M-2\", \"name\": \"Mug\", \"pence\": None},\n        {\"sku\": \"S-5\", \"name\": \"Spoon\", \"pence\": None}]")

### Exercise: Structured data first

Write `json_ld(html)`: every JSON object found in `<script type="application/ld+json">` blocks, in order. A block may hold one object or a list of them (add each), and a block whose text isn't valid JSON is skipped. Then `products(html)`: from those, the ones whose `@type` is `"Product"`, as `(name, price, currency)` tuples, where `price` is a `float` from the object's `offers` (a dict with `price` and `priceCurrency`; `price` may be a number or a string).

In [ ]:
import json
from html.parser import HTMLParser

def json_ld(html):
    ___

def products(html):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PAGE = \"\"\"<html><head>\n<script type=\"application/ld+json\">{\"@type\": \"Organization\", \"name\": \"Kettle Co\"}</script>\n<script>var notData = 1;</script>\n<script type=\"application/ld+json\">\n  [{\"@type\": \"Product\", \"name\": \"Kettle\", \"offers\": {\"price\": \"30.00\", \"priceCurrency\": \"GBP\"}},\n   {\"@type\": \"Product\", \"name\": \"Toaster\", \"offers\": {\"price\": 45.5, \"priceCurrency\": \"GBP\"}}]\n</script>\n<script type=\"application/ld+json\">{ this is broken </script>\n</head><body><h1>Kettles</h1></body></html>\"\"\"\n\ndef test_json_ld():\n    \"every block, lists flattened, broken ones skipped\"\n    found = json_ld(PAGE)\n    assert [x[\"name\"] for x in found] == [\"Kettle Co\", \"Kettle\", \"Toaster\"]\n\ndef test_products():\n    \"typed data\"\n    assert products(PAGE) == [(\"Kettle\", 30.0, \"GBP\"), (\"Toaster\", 45.5, \"GBP\")]")

### Exercise: A polite pace

Write a class `RateLimiter(min_interval, clock, sleep)`: `wait(host)` makes sure at least `min_interval` seconds pass between the **starts** of two requests to the same host. If not enough time has passed since the last request to that host, call `sleep` with exactly the time left; then record the current time (after any sleep) as that host's last request. Different hosts don't wait for each other. `set_interval(host, seconds)` overrides the interval for one host (for a `Crawl-delay`).

In [ ]:
class RateLimiter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_limiter():\n    \"waits only as long as needed\"\n    class Clock:\n        t = 100.0\n        def __call__(self):\n            return self.t\n        def sleep(self, s):\n            slept.append(s)\n            self.t += s\n    slept = []\n    clock = Clock()\n    rl = RateLimiter(2.0, clock, clock.sleep)\n    rl.wait(\"a.example\")\n    rl.wait(\"b.example\")\n    assert slept == []\n    clock.t += 0.5\n    rl.wait(\"a.example\")\n    assert slept == [1.5] and clock.t == 102.0\n    clock.t += 5\n    rl.wait(\"a.example\")\n    assert slept == [1.5]\n    rl.set_interval(\"b.example\", 10)\n    rl.wait(\"b.example\")\n    assert slept == [1.5, 3.0]")

### Bug hunt: Links glued together

Joining links with `+` gets a doubled slash for one and the wrong folder for the other, so this prints `https://news.example/world//about` and `https://news.example/world/../sport/`. Resolve them the way a browser does, so it prints `https://news.example/about` and `https://news.example/sport/`.

In [ ]:
page = "https://news.example/world/"
for href in ["/about", "../sport/"]:
    print(page + href)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"urljoin\"\n    expect_output(\"https://news.example/about\\nhttps://news.example/sport/\")")

### Bug hunt: A headline cut short

`handle_data` can be called several times for one element (here, once before the `<em>` and once inside it and once after), but the parser **replaces** the text each time, so it prints `'\n  on Mars'`. Collect all of it, and tidy the whitespace, so it prints `'Water found on Mars'`.

In [ ]:
from html.parser import HTMLParser

class Headline(HTMLParser):
    def __init__(self):
        super().__init__()
        self.inside, self.text = False, ""
    def handle_starttag(self, tag, attrs):
        if tag == "h1":
            self.inside = True
    def handle_endtag(self, tag):
        if tag == "h1":
            self.inside = False
    def handle_data(self, data):
        if self.inside:
            self.text = data

p = Headline()
p.feed("<h1>Water <em>found</em>\n  on Mars</h1>")
print(repr(p.text))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"add the pieces\"\n    expect_output(\"'Water found on Mars'\")")

### Bug hunt: The same page, again and again

This crawler only marks a page as seen **after** fetching it, and treats `item.html#reviews` as a different page from `item.html`, so pages reached by several links are fetched several times and it prints `fetched 6 pages`. Normalise each URL (drop the fragment) and mark it seen when it's **queued**, so it prints `fetched 3 pages`.

In [ ]:
from collections import deque

SITE = {
    "https://s.example/": ["https://s.example/item.html", "https://s.example/about.html", "https://s.example/item.html#reviews"],
    "https://s.example/item.html": ["https://s.example/", "https://s.example/about.html"],
    "https://s.example/about.html": ["https://s.example/item.html#top"],
}
fetched = []
queue, seen = deque(["https://s.example/"]), set()
while queue:
    url = queue.popleft()
    if url in seen:
        continue
    fetched.append(url)
    links = SITE.get(url.split("#")[0], [])
    seen.add(url)
    for link in links:
        if link not in seen:
            queue.append(link)
print(f"fetched {len(fetched)} pages")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"seen when queued, fragments dropped\"\n    expect_output(\"fetched 3 pages\")")

### Puzzle: One polite fetch

Arrange the steps a crawler takes for each page it takes off the queue. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # urljoin("https://a.example/x/y.html", "z.html")
q = ___   # urldefrag("https://a.example/p#top")[0]
r = ___   # the tag name handle_starttag gets for <A HREF="/">
s = ___   # the file a site uses to tell bots what to leave alone
t = ___   # the status code that means "slow down, too many requests"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"https://a.example/x/z.html\" and q == \"https://a.example/p\" and r == \"a\", \"p, q and r\"\n    assert s == \"robots.txt\" and t == 429, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a polite crawler

Write `crawl(start, fetch, robots_txt, agent, clock, sleep, max_pages=20)`. `fetch(url, headers)` returns `(status, headers, body)` (headers as a dict). Crawl breadth-first from `start`, staying on its host. Before each fetch, check `robots_txt` (with `urllib.robotparser`) and skip disallowed pages without fetching them, and wait with a per-host rate limit: `crawl_delay` from robots.txt, or `1.0` second if none (use `clock` and `sleep`; the first request doesn't wait). Send `{"User-Agent": agent}`. On `429` or `503`, sleep for the `Retry-After` header's seconds (or `30` if missing) and try the same page again, up to 2 retries; after that, give up on it. Parse only `200` responses whose `Content-Type` starts with `text/html`: record the page's `<title>` (whitespace collapsed, `""` if none) and queue its same-site links (resolved, without fragments, never queued twice). Stop after `max_pages` fetched pages (counting every call to `fetch`). Return a dict with `"titles"` (`{url: title}` for parsed pages), `"skipped"` (sorted URLs refused by robots.txt), `"failed"` (sorted URLs that ended with any status other than `200`) and `"fetches"` (the number of calls to `fetch`).

In [ ]:
from collections import deque
from html.parser import HTMLParser
from urllib.parse import urljoin, urldefrag, urlsplit
from urllib.robotparser import RobotFileParser

def crawl(start, fetch, robots_txt, agent, clock, sleep, max_pages=20):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _site():\n    H = {\"Content-Type\": \"text/html; charset=utf-8\"}\n    pages = {\n        \"https://s.example/\": (200, H, '<title>Home</title><a href=\"/a\">A</a> <a href=\"/b#x\">B</a> <a href=\"/private/1\">P</a> <a href=\"https://elsewhere.example/\">E</a> <a href=\"/data.json\">D</a>'),\n        \"https://s.example/a\": (200, H, '<title> Page\\n A </title><a href=\"/\">home</a><a href=\"/missing\">M</a><a href=\"/busy\">busy</a>'),\n        \"https://s.example/b\": (200, H, '<html><body>no title <a href=\"a\">A</a></body></html>'),\n        \"https://s.example/data.json\": (200, {\"Content-Type\": \"application/json\"}, '{\"a\": \"<a href=/zzz>\"}'),\n    }\n    calls, slept = [], []\n    class Clock:\n        t = 0.0\n        def __call__(self):\n            return self.t\n        def sleep(self, s):\n            slept.append(s)\n            self.t += s\n    clock = Clock()\n    busy = {\"n\": 0}\n    def fetch(url, headers):\n        calls.append((url, headers[\"User-Agent\"], clock.t))\n        clock.t += 0.25\n        if url == \"https://s.example/busy\":\n            busy[\"n\"] += 1\n            return (429, {\"Retry-After\": \"5\"}, \"\") if busy[\"n\"] < 3 else (200, H, \"<title>Busy</title>\")\n        return pages.get(url, (404, {\"Content-Type\": \"text/html\"}, \"not found\"))\n    return fetch, clock, calls, slept\n\nROBOTS = \"User-agent: *\\nDisallow: /private\\nCrawl-delay: 2\\n\"\n\ndef test_crawl():\n    \"titles, robots, failures and retries\"\n    fetch, clock, calls, slept = _site()\n    r = crawl(\"https://s.example/\", fetch, ROBOTS, \"AtlasBot/1.0\", clock, clock.sleep)\n    assert r[\"titles\"] == {\"https://s.example/\": \"Home\", \"https://s.example/a\": \"Page A\", \"https://s.example/b\": \"\",\n                           \"https://s.example/busy\": \"Busy\"}\n    assert r[\"skipped\"] == [\"https://s.example/private/1\"] and r[\"failed\"] == [\"https://s.example/missing\"]\n    assert r[\"fetches\"] == 8 and [c[0] for c in calls].count(\"https://s.example/busy\") == 3\n    assert all(agent == \"AtlasBot/1.0\" for _, agent, _ in calls)\n    times = [t for _, _, t in calls]\n    assert all(b - a >= 2 - 1e-9 for a, b in zip(times, times[1:])), \"at least 2 seconds between requests\"\n    assert 5 in slept or 5.0 in slept, \"Retry-After honoured\"\n\ndef test_limits():\n    \"the page limit, the default delay, and giving up\"\n    fetch, clock, calls, slept = _site()\n    r = crawl(\"https://s.example/\", fetch, \"\", \"Bot\", clock, clock.sleep, max_pages=3)\n    assert r[\"fetches\"] == 3 and len(calls) == 3\n    times = [t for _, _, t in calls]\n    assert abs((times[1] - times[0]) - 1.0) < 1e-9\n    H = {\"Content-Type\": \"text/html\"}\n    def always_busy(url, headers):\n        return 503, {}, \"\"\n    r = crawl(\"https://q.example/\", always_busy, \"\", \"Bot\", clock, clock.sleep)\n    assert r[\"fetches\"] == 3 and r[\"failed\"] == [\"https://q.example/\"] and r[\"titles\"] == {}")

## Warm-up

Links in a page are written **relative** to the page they're on, just as paths are relative to a folder. `urljoin` resolves them exactly as a browser would; string concatenation gets most of them wrong. `urldefrag` strips the `#fragment`, which only points within a page, so `item.html#top` and `item.html` are the same page.

## Should you scrape it?

Before writing a line of scraping code, check:

1. **Is there an API, a download or a feed?** Many sites publish their data properly: an API (Week 76), a CSV download, an RSS feed or a sitemap. Use that: it's faster, it won't break when the page design changes, and it's plainly allowed.
2. **The terms of use.** Some sites forbid automated access. Copyright and database rights can protect content even when it's public. Republishing someone else's content is a different matter from reading it for your own analysis.
3. **robots.txt.** At `https://SITE/robots.txt`, a site lists the parts automated visitors should leave alone, sometimes with a `Crawl-delay`. It isn't enforced by anything, but respecting it is the clearest sign of a well-behaved bot, and ignoring it is a good way to get blocked.
4. **Personal data.** Scraping information about people (names, contact details, profiles) brings in data protection law (in the UK and EU, the GDPR) whether or not the data was public. If you don't need it, don't collect it.
5. **Load.** A scraper can send requests far faster than any person. One request every second or two, at quiet times, is a reasonable starting point; parallel requests to a small site are not.

And always **identify yourself**: send a `User-Agent` header that names your bot and gives a way to contact you (`AtlasBot/1.0 (+https://example.org/bot)`), rather than pretending to be a browser.

**robots.txt** is a list of groups. Each starts with one or more `User-agent:` lines (`*` for everyone) followed by `Disallow:` and `Allow:` rules on path prefixes. A bot uses the group that names it, and otherwise the `*` group. Python reads it with `urllib.robotparser`:

`AtlasBot` has a group of its own, so the `*` group doesn't apply to it at all. Python's parser uses the **first** rule in the group that matches the path, so put specific `Allow` lines before broader `Disallow` lines when you write one. (Google's crawler uses the longest matching rule instead; well-written files work either way.)

## Parsing HTML

HTML is a tree of **tags** with **attributes** and **text**. The standard library's `html.parser.HTMLParser` reads it as a stream of events: subclass it and override the methods for the events you care about.

| Method | Called for | Gets |
|---|---|---|
| `handle_starttag(tag, attrs)` | `<a href="/x" class="btn">` | `"a"`, `[("href", "/x"), ("class", "btn")]` |
| `handle_endtag(tag)` | `</a>` | `"a"` |
| `handle_data(data)` | text between tags | the text, with `&amp;` already turned into `&` |
| `handle_startendtag(tag, attrs)` | `<img ... />` | (by default it calls `handle_starttag` then `handle_endtag`) |

Call `parser.feed(html)` (as often as you like, with pieces), then `parser.close()`. Tag and attribute names arrive in lower case; attribute values can be `None` (for `<input disabled>`).

The usual pattern is a small **state machine**: remember where you are (inside a `<title>`, inside a `<td>`, inside the product card), and collect text only while there. Text arrives in **pieces** (`<p>Fish <b>and</b> chips</p>` gives `"Fish "`, `"and"`, `" chips"`), so add pieces together and tidy the whitespace at the end with `" ".join(text.split())`.

Real pages are messy (missing closing tags, odd nesting), and `HTMLParser` copes with them without complaint. For bigger jobs, the third-party **Beautiful Soup** (`pip install beautifulsoup4`) builds the whole tree and lets you search it with CSS selectors (`soup.select("h2.headline")`), and **lxml** is faster still. The ideas are the same.

## Links and structured data

**Links.** Collect `href` values from `<a>` tags, then:

- resolve each with `urljoin(page_url, href)`;
- drop the fragment with `urldefrag`;
- skip links that aren't web pages: `mailto:`, `tel:` and `javascript:`;
- keep only `http` and `https` links, and, for a crawler, only those on the same host (`urlsplit(url).netloc`).

**Structured data.** Many pages carry their data in a machine-readable form too, for search engines: **JSON-LD** blocks, `<script type="application/ld+json">{ "@type": "Product", "name": ..., "offers": {...} }</script>`. Look for it before scraping visible text: it's tidier, typed, and much less likely to change when the page is redesigned. The same goes for `<meta property="og:title">` tags and sitemaps (`/sitemap.xml`, a list of every page the site wants found).

**Test with saved pages.** Save a copy of each kind of page you scrape (a "fixture") and test your parser against it. When the site changes and the scraper breaks, as it eventually will, you'll know at once which part, and you can fix it without hammering the site while you debug.

## Crawling politely

A **crawler** starts from one page and follows links to others. A polite one:

- keeps a **queue** of pages to visit and a **set** of pages already seen (by their normalised URL), so nothing is fetched twice;
- stays on the **site** it was asked to read, and within a **page limit**;
- checks **robots.txt** before every fetch;
- **waits** between requests to the same host (the `Crawl-delay`, or a second or two), using an injected clock and sleep function so the waiting is testable;
- treats **errors** as information: `404` means skip it; `429 Too Many Requests` and `503` mean slow down (honour a `Retry-After` header); repeated errors mean stop;
- only parses responses whose `Content-Type` is HTML;
- **caches** pages, and sends conditional requests (`If-None-Match` with the `ETag`, Week 76) when it revisits them, so unchanged pages cost almost nothing.

### On your own computer

Fetch real pages with `requests` (or `httpx`), always with a timeout and your own `User-Agent`:

```python
import requests
session = requests.Session()
session.headers["User-Agent"] = "AtlasBot/1.0 (+https://example.org/bot)"
response = session.get(url, timeout=10)
```

For sites that build their content with JavaScript, the HTML you fetch may be nearly empty: look in the browser's developer tools (Network tab) for the JSON API the page itself calls, which is usually the better source anyway. Browser-automation tools (Playwright, Selenium) can run the JavaScript for you, but they're heavy; reach for them last. Larger crawls are easier with the **Scrapy** framework, which has politeness settings (delays, robots.txt, concurrency limits) built in.

## Practice

## Quiz

## Daily challenge

## Before you go

Open the robots.txt of a few sites you use (add `/robots.txt` to their address) and see what they ask of bots; some have notes for people too. Tomorrow is the **automation project**: you'll combine the week into one tool that collects, tidies, archives and reports on a schedule.